In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:

import torch
import torch.nn as nn
import numpy as np

from datasets import Dataset
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments
    )

c:\Users\omole\anaconda3\envs\cudaenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# set device to cuda if it is available, else use cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
device

device(type='cuda')

In [5]:
with open("anna.txt", "r") as file:
    content = file.read()

In [6]:
import re

def clean_text(text):
    # convert into lower case letter
    text = text.lower()
    # 2. Add spaces around punctuation marks so they become separate tokens
    text = re.sub(r"([.,!?\"():;])", r" \1 ", text)
    # collapse multiple spaces into a single space
    text = re.sub(r"\s+", " ", text)
    tokens = text.strip().split()
    return " ".join(tokens)

cleaned_texts = clean_text(content)

In [7]:
cleaned_texts = cleaned_texts[:1024]

In [8]:
#### Defining model configuration and model selection
import os


MODEL_NAME = "gpt2"
# TXT_FILE_PATH = os.path.join(os.getcwd(),"anna.txt")
BLOCK_SIZE = 128
OUTPUT_DIR = "./finetuned_model"

In [9]:
# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [9]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("---> Tokenizing and chunking text into fixed sequences")

all_tokens = tokenizer(cleaned_texts, add_special_tokens=False)["input_ids"]

all_tokens

---> Tokenizing and chunking text into fixed sequences


[43582,
 352,
 3772,
 4172,
 389,
 477,
 12936,
 2162,
 790,
 19283,
 1641,
 318,
 19283,
 287,
 663,
 898,
 835,
 764,
 2279,
 373,
 287,
 10802,
 287,
 262,
 909,
 75,
 684,
 74,
 893,
 6,
 2156,
 764,
 262,
 3656,
 550,
 5071,
 326,
 262,
 5229,
 373,
 6872,
 319,
 281,
 38520,
 351,
 257,
 48718,
 2576,
 837,
 508,
 550,
 587,
 257,
 1089,
 408,
 287,
 511,
 1641,
 837,
 290,
 673,
 550,
 3414,
 284,
 607,
 5229,
 326,
 673,
 714,
 407,
 467,
 319,
 2877,
 287,
 262,
 976,
 2156,
 351,
 683,
 764,
 428,
 2292,
 286,
 9674,
 550,
 783,
 15436,
 1115,
 1528,
 837,
 290,
 407,
 691,
 262,
 5229,
 290,
 3656,
 2405,
 837,
 475,
 477,
 262,
 1866,
 286,
 511,
 1641,
 290,
 6641,
 837,
 547,
 32258,
 6921,
 286,
 340,
 764,
 790,
 1048,
 287,
 262,
 2156,
 2936,
 326,
 612,
 373,
 645,
 2565,
 287,
 511,
 2877,
 1978,
 837,
 290,
 326,
 262,
 28583,
 661,
 3181,
 1978,
 416,
 2863,
 287,
 597,
 3527,
 550,
 517,
 287,
 2219,
 351,
 530,
 1194,
 621,
 484,
 837,
 262,
 1866,
 286,
 262,
 

In [10]:
token_chunks = [
    all_tokens[i : i + BLOCK_SIZE]
    for i in range(0, len(all_tokens) - BLOCK_SIZE, BLOCK_SIZE)
]

In [11]:
from datasets import Dataset as HFDataset

In [12]:
dataset = HFDataset.from_dict({"input_ids": token_chunks})

print(f"Total training sequences created: {len(dataset)}")

Total training sequences created: 1


In [13]:
print(F"Loading pretrained model: {MODEL_NAME}.... --->>>")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)

Loading pretrained model: gpt2.... --->>>


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3700.00it/s]


In [14]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [15]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,per_device_train_batch_size=4,
    learning_rate=5e5,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy= "no",
    fp16=torch.cuda.is_available()
)

In [16]:
# Setting up trainer
trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=dataset,
    data_collator = data_collator
)

In [17]:
print("--> Fine-tuning model on custom dataset...")
trainer.train()

--> Fine-tuning model on custom dataset...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.634806
10,0.000000
15,0.000000


TrainOutput(global_step=15, training_loss=1.211602020263672, metrics={'train_runtime': 4.2409, 'train_samples_per_second': 3.537, 'train_steps_per_second': 3.537, 'total_flos': 979845120000.0, 'train_loss': 1.211602020263672, 'epoch': 15.0})

In [18]:
# Save final model and tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"--> Model saved to {OUTPUT_DIR}")

# ==========================================
# 5. Prediction / Inference
# ==========================================
print("\n" + "=" * 40)
print("  PREDICTION / TEXT GENERATION")
print("=" * 40)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.20s/it]

--> Model saved to ./finetuned_model

  PREDICTION / TEXT GENERATION


In [ ]:
def generate_next_words(prompt_text, max_new_tokens=30):
    

    model.eval()
    
    # Tokenize input prompt
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    
    # Generate text autoregressively
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode token IDs back to human-readable text
    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

# Test generation
seed_prompt = "Happy families are all"
generated_result = generate_next_words(seed_prompt, max_new_tokens=40)

print(f"\nPrompt: '{seed_prompt}'")
print(f"Generated Continuation:\n{generated_result}")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
